# Bybit intra-arb ETH — open / hedge aggregation

**Source:** `order_export_server` snapshot for `bybit-intra-arb01`, window `[03:00, 04:00) UTC 2026-05-29` (previous hour), file `uniform_orders.parquet`.

**Goal:** group every order by `from_key` so the open leg and its hedge land in one row.

- `from_key` links one open attempt to its hedge.
- `trading_venue` distinguishes the legs: `BybitMargin` = **open** (spot/margin LIMIT maker), `BybitFutures` = **hedge** (futures MARKET taker).
- Hedge is all-taker, so it is **unique** per `from_key` (0 or 1 futures row).

Section 8 cross-checks the latency timestamps against the exchange's ground-truth `execTime`.

In [1]:
import pandas as pd
import numpy as np
import json

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 220)

SNAPSHOT    = '20260529T030000.000000Z__20260529T040000.000000Z'  # [03:00,04:00) UTC
SRC         = f'{SNAPSHOT}/uniform_orders.parquet'  # snapshot dir is a subfolder of this env root
EXEC_JSON   = f'{SNAPSHOT}/bybit_exec_eth.json'     # exchange executions (see section 8)
OUT_PARQUET = f'{SNAPSHOT}/eth_open_hedge_agg.parquet'
OUT_CSV     = f'{SNAPSHOT}/eth_open_hedge_agg.csv'
SYMBOL      = 'ETHUSDT'
OPEN_VENUE  = 'BybitMargin'    # open leg  (maker LIMIT)
HEDGE_VENUE = 'BybitFutures'   # hedge leg (taker MARKET, unique)

# status lifecycle rank -> pick the terminal state of the open leg
STATUS_RANK = {'NEW': 0, 'PARTIALLY_FILLED': 1, 'EXPIRED': 2, 'CANCELED': 3, 'FILLED': 4}

## 1. Load & filter ETH

In [2]:
df = pd.read_parquet(SRC)
eth = df[df['symbol'] == SYMBOL].copy()
print(f'total rows={len(df)}  {SYMBOL} rows={len(eth)}')
print('columns:', list(eth.columns))
eth.head(3)

total rows=2672  ETHUSDT rows=1703
columns: ['key', 'ts_us', 'recv_ts_us', 'symbol', 'create_ts', 'update_ts', 'signal_ts', 'submit_ts', 'local_ts', 'mkt_ts', 'client_order_id', 'trading_venue', 'order_type', 'side', 'price', 'price_offset', 'amount_init', 'amount_update', 'status', 'from_key', 'from_key_hex']


,key,ts_us,recv_ts_us,symbol,create_ts,update_ts,signal_ts,submit_ts,local_ts,mkt_ts,client_order_id,trading_venue,order_type,side,price,price_offset,amount_init,amount_update,status,from_key,from_key_hex
0,00001780023906663783,1780023906663783,1780023906663783,ETHUSDT,1780023906662000,1780023906662000,1780023906659990,1780023906660669,1780023906663781,1780023906658000,3512128717969162241,BybitMargin,LIMIT,BUY,2001.86,1.135812e-16,0.02,0.0,NEW,3512128717969162241,33353132313238373137393639313632323431
1,00001780023910498723,1780023910498723,1780023910498723,ETHUSDT,1780023906662000,1780023910496000,1780023906659990,1780023910493836,1780023910498684,1780023910491000,3512128717969162241,BybitMargin,LIMIT,BUY,2001.86,1.135812e-16,0.02,0.0,CANCELED,3512128717969162241,33353132313238373137393639313632323431
2,00001780023914485984,1780023914485984,1780023914485984,ETHUSDT,1780023914483000,1780023914483000,1780023914480036,1780023914481150,1780023914485981,1780023914478000,3512128730854064129,BybitMargin,LIMIT,BUY,2002.05,0.000000e+00,0.02,0.0,NEW,3512128730854064129,33353132313238373330383534303634313239


## 2. from_key structure

Each `from_key` is one open attempt (a margin LIMIT BUY lifecycle). Confirm hedge uniqueness and the venue composition.

In [3]:
print('distinct from_key:', eth['from_key'].nunique())
print('null/empty from_key:', eth['from_key'].isna().sum(), (eth['from_key'] == '').sum())

comp = eth.groupby('from_key')['trading_venue'].agg(lambda s: tuple(sorted(s.unique())))
print('\nvenue-set composition across from_key groups:')
print(comp.value_counts())

per = eth.groupby(['from_key', 'trading_venue']).size().unstack(fill_value=0)
print('\n# hedge (futures) rows per from_key  -> confirms 0/1 uniqueness:')
print(per.get(HEDGE_VENUE, pd.Series(dtype=int)).value_counts().sort_index())
print('\n# open (margin) rows per from_key  -> lifecycle states of one order:')
print(per.get(OPEN_VENUE, pd.Series(dtype=int)).value_counts().sort_index())

distinct from_key: 745
null/empty from_key: 0 0

venue-set composition across from_key groups:
trading_venue
(BybitMargin,)                 530
(BybitFutures, BybitMargin)    215
Name: count, dtype: int64

# hedge (futures) rows per from_key  -> confirms 0/1 uniqueness:
BybitFutures
0    530
1    215
Name: count, dtype: int64

# open (margin) rows per from_key  -> lifecycle states of one order:
BybitMargin
1      7
2    734
3      3
4      1
Name: count, dtype: int64


In [4]:
# inspect one fully-hedged group: open NEW -> open FILLED -> hedge SELL MARKET FILLED
ex_key = comp[comp == (HEDGE_VENUE, OPEN_VENUE)].index[0]
cols = ['ts_us', 'trading_venue', 'side', 'order_type', 'price', 'amount_init', 'amount_update', 'status', 'client_order_id']
eth[eth['from_key'] == ex_key].sort_values('ts_us')[cols]

,ts_us,trading_venue,side,order_type,price,amount_init,amount_update,status,client_order_id
2,1780023914485984,BybitMargin,BUY,LIMIT,2002.05,0.02,0.00,NEW,3512128730854064129
3,1780023916406996,BybitMargin,BUY,LIMIT,2002.05,0.02,0.02,FILLED,3512128730854064129
4,1780023916414723,BybitFutures,SELL,MARKET,2001.16,0.02,0.02,FILLED,3512128713674194945


## 3. Aggregate per from_key

Collapse the open-leg lifecycle to its terminal state, attach the unique hedge leg. We also keep the **component timestamps** (for section 7) and each leg's **client_order_id** (= Bybit `orderLinkId`, for the section 8 exchange match).

`agg_group` always returns the same key set so every group yields an identical row schema.

In [5]:
TS_COLS = ['open_fill_ts', 'open_fill_exch_ts', 'open_first_ts', 'open_last_ts',
           'hedge_signal_ts', 'hedge_submit_ts', 'hedge_update_ts', 'hedge_ts']
ID_COLS = ['open_coid', 'hedge_coid']

def agg_group(g):
    op = g[g.trading_venue == OPEN_VENUE].sort_values('ts_us')
    hg = g[g.trading_venue == HEDGE_VENUE].sort_values('ts_us')

    # fixed schema with defaults so every group returns identical keys
    out = {
        'open_n_rows': len(op), 'open_n_coid': op.client_order_id.nunique() if len(op) else 0,
        'open_coid': None, 'hedge_coid': None,
        'open_side': None, 'open_type': None, 'open_status': None,
        'open_amt_init': np.nan, 'open_filled': np.nan,
        'open_fill_px': np.nan, 'open_fill_ts': np.nan, 'open_fill_exch_ts': np.nan,
        'open_first_ts': np.nan, 'open_last_ts': np.nan,
        'hedged': len(hg) > 0, 'hedge_n': len(hg),
        'hedge_side': None, 'hedge_type': None, 'hedge_status': None,
        'hedge_px': np.nan, 'hedge_amt': np.nan,
        'hedge_signal_ts': np.nan, 'hedge_submit_ts': np.nan,
        'hedge_update_ts': np.nan, 'hedge_ts': np.nan,
    }

    if len(op):
        term = op.loc[op.status.map(STATUS_RANK).idxmax()]          # terminal status row
        out['open_coid']     = str(int(op.client_order_id.iloc[0]))
        out['open_side']     = op.side.iloc[0]
        out['open_type']     = op.order_type.iloc[0]
        out['open_status']   = term.status
        out['open_amt_init'] = op.amount_init.max()
        out['open_filled']   = op.amount_update.max()
        out['open_first_ts'] = op.ts_us.iloc[0]
        out['open_last_ts']  = op.ts_us.iloc[-1]
        filled = op[op.status.isin(['FILLED', 'PARTIALLY_FILLED'])]
        if len(filled):
            out['open_fill_px']      = filled.price.iloc[-1]
            out['open_fill_ts']      = filled.ts_us.iloc[-1]        # spot fill received locally
            out['open_fill_exch_ts'] = filled.update_ts.iloc[-1]    # spot fill matched at exchange (system-recorded)

    if len(hg):
        h = hg.iloc[0]                                              # unique taker hedge
        out['hedge_coid']      = str(int(h.client_order_id))
        out['hedge_side']      = h.side
        out['hedge_type']      = h.order_type
        out['hedge_status']    = h.status
        out['hedge_px']        = h.price
        out['hedge_amt']       = h.amount_update
        out['hedge_signal_ts'] = h.signal_ts                       # == open_fill_exch_ts (verified)
        out['hedge_submit_ts'] = h.submit_ts                       # hedge order sent to exchange
        out['hedge_update_ts'] = h.update_ts                       # hedge matched at exchange (system, ms-rounded)
        out['hedge_ts']        = h.ts_us                           # hedge fill received locally

    return pd.Series(out)


agg = eth.groupby('from_key').apply(agg_group, include_groups=False).reset_index()
for c in TS_COLS:
    agg[c] = agg[c].astype('Int64')      # ts safe in float range; coids kept as exact strings
print('aggregation shape:', agg.shape)
agg.head()

aggregation shape: (745, 26)


,from_key,open_n_rows,open_n_coid,open_coid,hedge_coid,open_side,open_type,open_status,open_amt_init,open_filled,open_fill_px,open_fill_ts,open_fill_exch_ts,open_first_ts,open_last_ts,hedged,hedge_n,hedge_side,hedge_type,hedge_status,hedge_px,hedge_amt,hedge_signal_ts,hedge_submit_ts,hedge_update_ts,hedge_ts
0,3512128717969162241,2,1,3512128717969162241,NaN,BUY,LIMIT,CANCELED,0.02,0.00,NaN,<NA>,<NA>,1780023906663783,1780023910498723,False,0,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
1,3512128730854064129,2,1,3512128730854064129,3512128713674194945,BUY,LIMIT,FILLED,0.02,0.02,2002.05,1780023916406996,1780023916403000,1780023914485984,1780023916406996,True,1,SELL,MARKET,FILLED,2001.16,0.02,1780023916403000,1780023916407242,1780023916411000,1780023916414723
2,3512128743738966017,2,1,3512128743738966017,NaN,BUY,LIMIT,CANCELED,0.02,0.00,NaN,<NA>,<NA>,1780023919563700,1780023977847661,False,0,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
3,3512128748033933313,2,1,3512128748033933313,NaN,BUY,LIMIT,CANCELED,0.02,0.00,NaN,<NA>,<NA>,1780023919563649,1780023943548991,False,0,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>
4,3512128760918835201,2,1,3512128760918835201,NaN,BUY,LIMIT,CANCELED,0.02,0.00,NaN,<NA>,<NA>,1780023924804719,1780023928528807,False,0,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>


## 4. Derived metrics

For groups where both legs filled: hedge latency (open fill -> hedge fill) and the basis captured (hedge SELL price minus open BUY price).

In [6]:
agg['hedge_latency_us'] = (agg['hedge_ts'] - agg['open_fill_ts']).astype('Int64')
agg['spread_abs'] = agg['hedge_px'] - agg['open_fill_px']            # sell_future - buy_spot
agg['spread_bps'] = agg['spread_abs'] / agg['open_fill_px'] * 1e4
agg.head()

,from_key,open_n_rows,open_n_coid,open_coid,hedge_coid,open_side,open_type,open_status,open_amt_init,open_filled,open_fill_px,open_fill_ts,open_fill_exch_ts,open_first_ts,open_last_ts,hedged,hedge_n,hedge_side,hedge_type,hedge_status,hedge_px,hedge_amt,hedge_signal_ts,hedge_submit_ts,hedge_update_ts,hedge_ts,hedge_latency_us,spread_abs,spread_bps
0,3512128717969162241,2,1,3512128717969162241,NaN,BUY,LIMIT,CANCELED,0.02,0.00,NaN,<NA>,<NA>,1780023906663783,1780023910498723,False,0,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
1,3512128730854064129,2,1,3512128730854064129,3512128713674194945,BUY,LIMIT,FILLED,0.02,0.02,2002.05,1780023916406996,1780023916403000,1780023914485984,1780023916406996,True,1,SELL,MARKET,FILLED,2001.16,0.02,1780023916403000,1780023916407242,1780023916411000,1780023916414723,7727,-0.89,-4.445443
2,3512128743738966017,2,1,3512128743738966017,NaN,BUY,LIMIT,CANCELED,0.02,0.00,NaN,<NA>,<NA>,1780023919563700,1780023977847661,False,0,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
3,3512128748033933313,2,1,3512128748033933313,NaN,BUY,LIMIT,CANCELED,0.02,0.00,NaN,<NA>,<NA>,1780023919563649,1780023943548991,False,0,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN
4,3512128760918835201,2,1,3512128760918835201,NaN,BUY,LIMIT,CANCELED,0.02,0.00,NaN,<NA>,<NA>,1780023924804719,1780023928528807,False,0,NaN,NaN,NaN,NaN,NaN,<NA>,<NA>,<NA>,<NA>,<NA>,NaN,NaN


## 5. Summary

In [7]:
print('=== hedged ratio ===')
print(agg['hedged'].value_counts())

print('\n=== open_status x hedged ===')
print(agg.groupby(['open_status', 'hedged']).size())

filled_open = agg[agg.open_status == 'FILLED']
print(f'\nFILLED open groups={len(filled_open)}, hedged={int(filled_open.hedged.sum())}')
print('open_n_coid distribution (rows per from_key share one client_order_id):')
print(agg.open_n_coid.value_counts())

=== hedged ratio ===
hedged
False    530
True     215
Name: count, dtype: int64

=== open_status x hedged ===
open_status       hedged
CANCELED          False     520
EXPIRED           False       2
FILLED            True      215
NEW               False       7
PARTIALLY_FILLED  False       1
dtype: int64

FILLED open groups=215, hedged=215
open_n_coid distribution (rows per from_key share one client_order_id):
open_n_coid
1    745
Name: count, dtype: int64


In [8]:
hh = agg[agg.hedged].copy()
print('=== hedge latency (us): open fill -> hedge fill ===')
print(hh['hedge_latency_us'].astype('float').describe(percentiles=[.5, .9, .99]).round(1))
print('\n=== basis spread bps (hedge_sell - open_buy) ===')
print(hh['spread_bps'].describe(percentiles=[.5, .9, .99]).round(2))

=== hedge latency (us): open fill -> hedge fill ===
count       215.0
mean      11329.2
std       19726.9
min        5155.0
50%        6377.0
90%       12265.8
99%      110213.0
max      112326.0
Name: hedge_latency_us, dtype: float64

=== basis spread bps (hedge_sell - open_buy) ===
count    215.00
mean      -4.13
std        0.38
min       -5.04
50%       -4.09
90%       -3.69
99%       -3.36
max       -3.19
Name: spread_bps, dtype: float64


## 6. 对冲延迟详析（open fill → hedge fill）

现货腿成交回报到期货市价对冲成交之间的耗时。只在双腿都成交的 215 组上计算，单位微秒(us)，换算成毫秒(ms)看更直观。

In [9]:
# 只取已对冲的组；hedge_latency_us = hedge_ts(期货成交) - open_fill_ts(现货成交)
lat_us = agg.loc[agg.hedged, 'hedge_latency_us'].astype('float')
lat_ms = lat_us / 1000.0                                  # us -> ms

q = lat_ms.quantile([.5, .9, .99])
print('对冲延迟分位数 (ms):')
print(f'  样本数 n     : {int(lat_us.size)}')
print(f'  中位数 p50   : {q[.5]:.2f} ms   <- 典型水平')
print(f'  p90          : {q[.9]:.2f} ms')
print(f'  p99          : {q[.99]:.2f} ms')
print(f'  最大 max     : {lat_ms.max():.2f} ms')
print(f'  均值 mean    : {lat_ms.mean():.2f} ms   <- 被尾部少数慢单拉高, 不代表典型值')

# 分桶看分布: 绝大多数应落在 <10ms
buckets = pd.cut(lat_ms, bins=[0, 6, 10, 20, 50, 1e9],
                 labels=['<6ms', '6-10ms', '10-20ms', '20-50ms', '>50ms'], right=False)
print('\n延迟分桶 (笔数 / 占比):')
dist = buckets.value_counts().sort_index()
for k, v in dist.items():
    print(f'  {k:>8} : {v:>3}  ({v / lat_us.size:.1%})')

对冲延迟分位数 (ms):
  样本数 n     : 215
  中位数 p50   : 6.38 ms   <- 典型水平
  p90          : 12.27 ms
  p99          : 110.21 ms
  最大 max     : 112.33 ms
  均值 mean    : 11.33 ms   <- 被尾部少数慢单拉高, 不代表典型值

延迟分桶 (笔数 / 占比):
      <6ms :  82  (38.1%)
    6-10ms : 108  (50.2%)
   10-20ms :  15  (7.0%)
   20-50ms :   1  (0.5%)
     >50ms :   9  (4.2%)


In [10]:
# 尾部慢单清单: 用 50ms 作阈值 (约 p99 水平), 单独拎出来排查
SLOW_MS = 50.0
slow = hh[(hh['hedge_latency_us'].astype('float') / 1000.0) > SLOW_MS].copy()
slow['latency_ms'] = (slow['hedge_latency_us'].astype('float') / 1000.0).round(2)
print(f'慢对冲 (>{SLOW_MS:.0f}ms) 共 {len(slow)} 笔, 占已对冲 {len(slow) / len(hh):.1%}:')
print(slow.sort_values('latency_ms', ascending=False)
          [['from_key', 'open_fill_ts', 'hedge_ts', 'latency_ms', 'spread_bps']]
          .to_string(index=False))

# 结论:
#   - 对冲路径整体很稳: 绝大多数在 ~6ms 内完成 (现货成交回报 -> 立即市价吃单对冲);
#   - 中位 6.4ms / p90 12ms, 用中位数描述典型水平最合适;
#   - p99/max 跳到 ~110ms, 是少数尾部慢单 (回报抖动 / 限频 / 网络毛刺) 拉高了均值到 ~11ms;
#   - 这些慢单的 spread_bps 与整体无明显差异, 说明慢对冲未必吃亏更多, 但仍是延迟优化的排查目标。

慢对冲 (>50ms) 共 9 笔, 占已对冲 4.2%:
           from_key     open_fill_ts         hedge_ts  latency_ms  spread_bps
3512131389438820353 1780024815518452 1780024815630778      112.33   -4.487883
3512131333604245505 1780024815518826 1780024815630809      111.98   -4.487883
3512131350784114689 1780024815518706 1780024815630478      111.77   -4.487883
3512130654999412737 1780024622534670 1780024622635306      100.64   -4.088552
3512130624934641665 1780024622535159 1780024622635629      100.47   -4.088552
3512130650704445441 1780024622534951 1780024622635372      100.42   -4.088552
3512130642114510849 1780024622535061 1780024622635348      100.29   -4.088552
3512130646409478145 1780024622535242 1780024622635360      100.12   -4.088552
3512130607754772481 1780024622535323 1780024622635383      100.06   -4.088552


## 7. 延迟构成分解（component analysis）

把端到端对冲延迟拆成首尾相接、可严格相加的几段，分别看分位数，定位瓶颈在哪一环。

**关键身份关系（已在 215 组上严格验证，逐笔相等）：** `hedge.signal_ts ≡ open.update_ts`，即对冲信号的时间戳被回填到**现货成交在交易所撮合的时刻**。因此可以现货成交在交易所撮合的时刻 `open_fill_exch_ts` 为起点 `t0`，把总延迟拆成 4 段：

| 分量 | 含义 | 计算 |
|---|---|---|
| **C1 现货回报回传** | 现货成交@交易所 → 本地收到回报 | `open_fill_ts − open_fill_exch_ts` |
| **C2 本地反应发单** | 本地收到现货成交 → 对冲单发出 | `hedge_submit_ts − open_fill_ts` |
| **C3 往返+撮合** | 对冲单发出 → 对冲在交易所成交 | `hedge_update_ts − hedge_submit_ts` |
| **C4 对冲回报回传** | 对冲成交@交易所 → 本地收到回报 | `hedge_ts − hedge_update_ts` |

派生总量：**敞口窗口** `= C1+C2+C3`（真正裸露单边敞口的时长，现货成交@交易所 → 对冲成交@交易所）；**全链路** `= C1+C2+C3+C4`（含对冲回报回传到本地）。

> 注意：`hedge_update_ts` 在系统侧按**毫秒取整**，故 C3 / C4 各自带约 ±1ms 的量化误差（相加抵消，不影响全链路）。section 8 用交易所真实 `execTime` 进一步复核这些时间戳。

In [11]:
# 只取双腿都成交、且各分量时间戳齐全的组
cc = agg[agg.hedged & agg.open_fill_exch_ts.notna()].copy()

# 4 个首尾相接的分量 (us)
cc['C1_现货回报回传'] = (cc.open_fill_ts    - cc.open_fill_exch_ts)
cc['C2_本地反应发单'] = (cc.hedge_submit_ts - cc.open_fill_ts)
cc['C3_往返撮合']     = (cc.hedge_update_ts - cc.hedge_submit_ts)
cc['C4_对冲回报回传'] = (cc.hedge_ts        - cc.hedge_update_ts)
# 派生总量
cc['敞口窗口']        = (cc.hedge_update_ts - cc.open_fill_exch_ts)   # C1+C2+C3
cc['全链路']          = (cc.hedge_ts        - cc.open_fill_exch_ts)   # C1+C2+C3+C4

COMPONENTS = ['C1_现货回报回传', 'C2_本地反应发单', 'C3_往返撮合', 'C4_对冲回报回传']
# 严格相加校验: 4 段之和必须等于全链路 (误差 0)
sum_err = int((cc[COMPONENTS].astype('float').sum(axis=1) - cc['全链路'].astype('float')).abs().max())
neg = {c: int((cc[c].astype('float') < 0).sum()) for c in COMPONENTS}
print(f'样本数 n = {len(cc)} | 相加误差(us) = {sum_err} | 负值分量 = {neg}')

样本数 n = 215 | 相加误差(us) = 0 | 负值分量 = {'C1_现货回报回传': 0, 'C2_本地反应发单': 0, 'C3_往返撮合': 0, 'C4_对冲回报回传': 0}


In [12]:
def pct_table_ms(frame, cols):
    """逐列输出分位数统计 (us -> ms)。"""
    rows = {}
    for c in cols:
        s = frame[c].astype('float') / 1000.0
        rows[c] = {'p50': s.quantile(.5), 'p90': s.quantile(.9), 'p99': s.quantile(.99),
                   'mean': s.mean(), 'max': s.max()}
    return pd.DataFrame(rows).T.round(2)

tbl = pct_table_ms(cc, COMPONENTS + ['敞口窗口', '全链路'])
print('各分量延迟分位数 (ms), n =', len(cc))
print(tbl.to_string())

# 各分量对全链路的中位贡献占比
med = (cc[COMPONENTS].astype('float').median())
print('\n各分量中位数占全链路中位数比重:')
for c in COMPONENTS:
    print(f'  {c:14} {med[c] / 1000.0:7.2f} ms  ({med[c] / med.sum():5.1%})')

各分量延迟分位数 (ms), n = 215
             p50    p90     p99   mean     max
C1_现货回报回传   3.48   5.97  110.15   7.52  110.32
C2_本地反应发单   0.12   0.25    0.26   0.16    0.29
C3_往返撮合     2.89   3.78    6.69   3.16   29.20
C4_对冲回报回传   3.10   7.47  107.18   8.01  109.48
敞口窗口        7.00  10.60  113.00  10.83  114.00
全链路        10.14  17.48  210.37  18.85  210.63

各分量中位数占全链路中位数比重:
  C1_现货回报回传         3.48 ms  (36.3%)
  C2_本地反应发单         0.12 ms  ( 1.3%)
  C3_往返撮合           2.89 ms  (30.1%)
  C4_对冲回报回传         3.10 ms  (32.3%)


In [13]:
# 结论 (各分量分位数解读):
#   C2 本地反应发单 : p50 0.12ms / p99 0.26ms  —— 自有链路极快且稳, 不是瓶颈;
#   C3 往返+撮合    : p50 2.89ms / p99 6.7ms    —— 对冲单到交易所的往返+撮合, 表现稳定;
#   C1 现货回报回传 : p50 3.48ms / p99 110ms    —— 现货成交回报入站, 中位健康但尾部极长;
#   C4 对冲回报回传 : p50 3.10ms / p99 107ms    —— 对冲成交回报入站, 同样中位健康但尾部极长。
# 全链路 p99 ~210ms 的尾部几乎全部来自 C1 + C4 这两段"交易所->本地回报回传",
# 而非我们自己的决策(C2)或撮合(C3); 与第6节发现的"并发批量成交时回报排队"一致 ——
# 优化重点应放在回报通道(WS 回报解析/排队/限频), 而不是下单链路。
print('see comments above')

see comments above


## 8. 交易所成交时间核对（execTime vs 系统记录 filled time）

用交易所真实成交时刻 `execTime` 复核 section 7 里当作"交易所撮合时刻"使用的两个**系统记录时间戳** (`open_fill_exch_ts` / `hedge_update_ts`)，看延迟结论是否成立。

**数据来源：** `bybit_exec_eth.json` 由 `scripts/fetch_bybit_exec.py` 在 SG 机器 (`ubuntu@47.131.162.78`，`bybit-intra-arb01` env，`source env.sh` 取凭证) 调 Bybit `GET /v5/execution/list` 拉取，窗口 `[03:00,04:00) UTC`，category `spot`(开仓) + `linear`(对冲)，逐笔 `execId` 粒度。

**匹配键：** `orderLinkId == 本地 client_order_id`（开仓用 `open_coid`，对冲用 `hedge_coid`）—— 此前已验证开/对冲两腿各 100% 命中、零缺失。一个订单可拆多笔成交，取**末笔 `execTime`**（= 完全成交时刻）对齐本地的 `update_ts`。

In [14]:
# 载入交易所逐笔成交, 按 orderLinkId rollup 出末笔成交时刻 (ms, UTC)
ex = pd.DataFrame(json.load(open(EXEC_JSON)))
ex['execTime'] = pd.to_numeric(ex['execTime'])              # 交易所撮合时刻 (ms)
ex['orderLinkId'] = ex['orderLinkId'].astype('int64')       # == 本地 client_order_id
exo = (ex.groupby(['_category', 'orderLinkId'])
         .agg(exch_first_ms=('execTime', 'min'),
              exch_last_ms=('execTime', 'max'),
              n_exec=('execId', 'size')).reset_index())
spot_last = exo.loc[exo._category == 'spot'].set_index('orderLinkId')['exch_last_ms']
lin_last  = exo.loc[exo._category == 'linear'].set_index('orderLinkId')['exch_last_ms']
print('交易所逐笔成交: 总 %d 笔 | spot 订单=%d, linear 订单=%d'
      % (len(ex), (exo._category == 'spot').sum(), (exo._category == 'linear').sum()))
print('多笔成交订单: spot=%d, linear=%d'
      % ((exo[exo._category=='spot'].n_exec > 1).sum(), (exo[exo._category=='linear'].n_exec > 1).sum()))

交易所逐笔成交: 总 506 笔 | spot 订单=216, linear 订单=215
多笔成交订单: spot=4, linear=70


In [15]:
# 把交易所末笔 execTime 映射到聚合表 (开仓 via open_coid -> spot, 对冲 via hedge_coid -> linear)
x = cc.copy()
x['o_exch_ms'] = x['open_coid'].astype('int64').map(spot_last)
x['h_exch_ms'] = x['hedge_coid'].astype('int64').map(lin_last)
x = x.dropna(subset=['o_exch_ms', 'h_exch_ms'])

# 偏移: 系统记录成交时间(update_ts) - 交易所真实 execTime, 单位统一到 ms
x['d_open']  = x['open_fill_exch_ts'].astype('float') / 1000.0 - x['o_exch_ms']
x['d_hedge'] = x['hedge_update_ts'].astype('float')   / 1000.0 - x['h_exch_ms']

def q_ms(s):
    return {'p50': s.quantile(.5), 'p90': s.quantile(.9), 'p99': s.quantile(.99),
            'mean': s.mean(), 'min': s.min(), 'max': s.max()}

off = pd.DataFrame({'开仓 d = 系统-交易所 (ms)': q_ms(x['d_open']),
                    '对冲 d = 系统-交易所 (ms)': q_ms(x['d_hedge'])}).T.round(2)
print('系统记录成交时间(update_ts) vs 交易所 execTime 偏移, n =', len(x))
print(off.to_string())

系统记录成交时间(update_ts) vs 交易所 execTime 偏移, n = 215
                    p50  p90  p99  mean  min  max
开仓 d = 系统-交易所 (ms)  2.0  2.0  3.0  1.86  1.0  4.0
对冲 d = 系统-交易所 (ms)  2.0  2.0  3.0  1.57  1.0  3.0


In [16]:
# 敞口窗口两种口径对比: 系统(update_ts 差) vs 交易所(execTime 差)
x['expo_mine'] = (x['hedge_update_ts'].astype('float') - x['open_fill_exch_ts'].astype('float')) / 1000.0
x['expo_exch'] = x['h_exch_ms'] - x['o_exch_ms']

cmp = pd.DataFrame({'系统口径 (update_ts 差)': q_ms(x['expo_mine']),
                    '交易所口径 (execTime 差)': q_ms(x['expo_exch'])}).T.round(2)
print('敞口窗口 现货成交 -> 对冲成交 (ms), n =', len(x))
print(cmp.to_string())

delta = x['expo_exch'] - x['expo_mine']
print(f'\ndelta(交易所 - 系统): p50={delta.quantile(.5):.2f}  mean={delta.mean():.2f}  std={delta.std():.2f} ms')

敞口窗口 现货成交 -> 对冲成交 (ms), n = 215
                    p50   p90     p99   mean  min    max
系统口径 (update_ts 差)  7.0  10.6  113.00  10.83  5.0  114.0
交易所口径 (execTime 差)  7.0  11.0  113.86  11.12  5.0  114.0

delta(交易所 - 系统): p50=0.00  mean=0.28  std=0.74 ms


In [17]:
# 结论 (交易所 execTime 复核):
#   1) 系统记录的成交时间 update_ts 比交易所真实 execTime 稳定地晚约 2ms
#      (开仓/对冲两腿一致: p50≈2ms, p99≈3ms, max≤4ms) —— 恒定系统性偏移,
#      大概率是回报落库/事件时间口径差, 不是抖动。
#   2) 该偏移在"现货成交->对冲成交"敞口窗口里两腿同号抵消: 两种口径几乎重合
#      (p50 均 7.00ms; delta p50=0.00, mean≈0.28ms, std≈0.74ms)。
#   => 延迟结论不变: 敞口窗口与延迟分解(section 7)在交易所真值下成立。
#      唯一修正: 绝对成交时间戳带 ~2ms 系统延迟, 会让 C3(submit->match)略高估、
#      C4(match->recv)略低估各约 2ms, 但全链路总延迟与敞口窗口不受影响。
print('see comments above')

see comments above


## 9. 指定订单查找与开仓/对冲配对追溯

按交易所 `orderId` 片段查找订单，并用 `orderLinkId == 本地 client_order_id` + 本地 `from_key` 在现货(spot/开仓)与合约(linear/对冲)之间**双向追溯**，判定给定订单是否构成配对关系。

**匹配链路（全程精确键，不靠时间戳）：**
`合约 orderId(UUID) ⇄ orderLinkId(=本地 client_order_id) ⇄ from_key ⇄ 现货 orderLinkId ⇄ 现货 orderId`

**待查订单：**
- 现货 (spot `orderId` 末8位)：`12673280`, `47987968`
- 合约 (linear `orderId` 片段)：`26e22620`, `a9b02672`

In [18]:
from datetime import datetime, timezone

def ms2utc(ms):
    return datetime.fromtimestamp(ms / 1000, tz=timezone.utc).strftime('%H:%M:%S.%f')[:-3] + 'Z'

# `ex` (交易所逐笔成交) 已在 section 8 载入; execTime/execQty/orderLinkId 已转好类型。
def find_exec(category, frag=None, order_link_id=None):
    """按 category 过滤交易所成交, 可按 orderId 片段或 orderLinkId 精确取。"""
    d = ex[ex._category == category]
    if frag is not None:
        d = d[d.orderId.astype(str).str.contains(str(frag), regex=False)]
    if order_link_id is not None:
        d = d[d.orderLinkId == int(order_link_id)]
    return d.sort_values('execTime')

# orderLinkId(=client_order_id) -> from_key, 用精确 int64 (避开 float 精度坑)
olid_to_fk = (eth.assign(_coid=eth.client_order_id.astype('int64'))
                 .drop_duplicates('_coid').set_index('_coid')['from_key'])

def trace_pair(fk):
    """给定 from_key, 汇总现货(开仓)与合约(对冲)两腿的交易所成交。"""
    op = eth[(eth.from_key == fk) & (eth.trading_venue == OPEN_VENUE)
             & (eth.status.isin(['FILLED', 'PARTIALLY_FILLED']))]
    hg = eth[(eth.from_key == fk) & (eth.trading_venue == HEDGE_VENUE)]
    o_olid, h_olid = int(op.client_order_id.iloc[0]), int(hg.client_order_id.iloc[0])
    se, he = find_exec('spot', order_link_id=o_olid), find_exec('linear', order_link_id=h_olid)
    return {
        'from_key': fk,
        'spot_orderLinkId': o_olid, 'spot_orderId': se.orderId.iloc[0],
        'spot_tail': str(se.orderId.iloc[0])[-8:],
        'spot_time': ms2utc(se.execTime.max()), 'spot_qty': float(se.execQty.sum()),
        'fut_orderLinkId': h_olid, 'fut_orderId': he.orderId.iloc[0],
        'fut_frag': str(he.orderId.iloc[0])[-8:],
        'fut_time': ms2utc(he.execTime.max()), 'fut_qty': float(he.execQty.sum()),
        'latency_ms': int(he.execTime.max() - se.execTime.max()),
    }

print('helpers ready; exchange execs in scope:', len(ex))

helpers ready; exchange execs in scope: 506


In [19]:
# 解析给定订单 -> (orderLinkId, from_key)
GIVEN_SPOT_TAILS = ['12673280', '47987968']   # spot orderId 末8位
GIVEN_FUT_FRAGS  = ['26e22620', 'a9b02672']   # linear orderId 片段

def resolve(category, frag):
    d = find_exec(category, frag=frag)
    if d.empty:
        return None
    olid = int(d.orderLinkId.iloc[0])
    return {'frag': frag, 'category': category, 'orderLinkId': olid,
            'orderId': d.orderId.iloc[0], 'from_key': olid_to_fk.get(olid)}

spots = [resolve('spot', t) for t in GIVEN_SPOT_TAILS]
futs  = [resolve('linear', f) for f in GIVEN_FUT_FRAGS]
print('现货解析:')
for s in spots:
    print(f"  {s['frag']}  orderId={s['orderId']}  orderLinkId={s['orderLinkId']}  from_key={s['from_key']}")
print('合约解析:')
for h in futs:
    print(f"  {h['frag']}  orderId={h['orderId']}  orderLinkId={h['orderLinkId']}  from_key={h['from_key']}")

# 按涉及到的 from_key 做配对追溯
fks = list(dict.fromkeys([r['from_key'] for r in spots + futs if r]))
pair = pd.DataFrame([trace_pair(fk) for fk in fks])
print('\n=== 开仓/对冲配对追溯 ===')
print(pair[['from_key', 'spot_tail', 'spot_time', 'fut_frag', 'fut_time',
            'latency_ms', 'spot_qty', 'fut_qty']].to_string(index=False))

现货解析:


  12673280  orderId=2225411990212673280  orderLinkId=3512134103858151425  from_key=3512134103858151425
  47987968  orderId=2225412032247987968  orderLinkId=3512134121038020609  from_key=3512134121038020609
合约解析:
  26e22620  orderId=4a40386f-0f32-4ae2-a2e4-3e0926e22620  orderLinkId=3512128713674195076  from_key=3512134103858151425
  a9b02672  orderId=13a14bf4-16c7-486e-8327-7b31a9b02672  orderLinkId=3512128713674195075  from_key=3512134121038020609

=== 开仓/对冲配对追溯 ===


           from_key spot_tail     spot_time fut_frag      fut_time  latency_ms  spot_qty  fut_qty
3512134103858151425  12673280 03:36:28.843Z 26e22620 03:36:28.852Z           9      0.02     0.02
3512134121038020609  47987968 03:36:28.843Z a9b02672 03:36:28.852Z           9      0.02     0.02


In [20]:
# 配对关系判定: 给定的现货与合约是否一一匹配
fut_by_fk = {h['from_key']: h for h in futs if h}
print('=== 配对关系判定 ===')
pairs = []
for s in spots:
    h = fut_by_fk.get(s['from_key'])
    if h:
        pairs.append((s['frag'], h['frag'], s['from_key']))
        print(f"  现货 {s['frag']}  <-->  合约 {h['frag']}   (同一 from_key {s['from_key']})")
    else:
        print(f"  现货 {s['frag']}  对冲腿不在给定合约列表内")

all_matched = len(pairs) == len(GIVEN_SPOT_TAILS) == len(GIVEN_FUT_FRAGS)
qty_ok = (pair['spot_qty'] == pair['fut_qty']).all()
print(f"\n两两配对成立: {all_matched} | 两腿数量一致(open==hedge): {qty_ok}")

# 结论:
#   - 给定两组订单构成配对关系, 且是两两 1:1 (各自独立 from_key, 不是交叉/合并):
#       12673280 <-> 26e22620 , 47987968 <-> a9b02672
#   - 每个现货开仓有专属对冲合约 (各 0.02, open BUY spot / hedge SELL linear);
#   - 两对恰在同一并发批次成交 (现货同在 ...843Z, 合约同在 ...852Z), 各 9ms;
#   - 配对全程靠精确键 (orderLinkId/from_key/UUID 互查), 与时间戳无关。
print('配对:', ', '.join(f"{a}<->{b}" for a, b, _ in pairs))

=== 配对关系判定 ===
  现货 12673280  <-->  合约 26e22620   (同一 from_key 3512134103858151425)
  现货 47987968  <-->  合约 a9b02672   (同一 from_key 3512134121038020609)

两两配对成立: True | 两腿数量一致(open==hedge): True
配对: 12673280<->26e22620, 47987968<->a9b02672


## 10. 开仓即时边际 — 亏损来源拆解（entry-instant edge）

口径：只看开仓这一脚锁进的即时盈亏（不含资金费/平仓）。全部用**交易所真实成交**（VWAP + 真实 `execFee`），逐对(by `from_key`)拆解：

`入场净边际(USDT) = Q·(对冲卖价_perp − 开仓买价_spot) − (Σ对冲手续费 + Σ开仓手续费)`

拆成两块（均对名义额取 bps，负=亏）：
- **价差 price_bps** = `Q·(h_vwap − o_vwap) / 名义额`：买现货 vs 卖 perp 的已实现基差。
- **手续费 fee_bps** = `−(Σexec_fee) / 名义额`：spot maker rebate(−0.5bp) − linear taker(+1.5bp)。

`execFee` 符号：正=支付，负=返还(rebate)。

In [21]:
# 交易所真实成交按订单 rollup (VWAP / 数量 / 手续费 / 名义额)
for c in ['execPrice', 'execQty', 'execFee', 'execValue']:
    ex[c] = pd.to_numeric(ex[c], errors='coerce')

def _roll(g):
    q = g.execQty.sum()
    return pd.Series({'qty': q, 'vwap': (g.execPrice * g.execQty).sum() / q,
                      'fee': g.execFee.sum(), 'notional': g.execValue.sum(),
                      'feeRate': float(g.feeRate.iloc[0]), 'isMaker': bool(g.isMaker.iloc[0])})

spot_roll = ex[ex._category == 'spot'].groupby('orderLinkId').apply(_roll, include_groups=False)
lin_roll  = ex[ex._category == 'linear'].groupby('orderLinkId').apply(_roll, include_groups=False)

# 用本地 from_key 配对两腿 (open_coid/hedge_coid 已是精确字符串)
pairs = []
for _, r in agg[agg.hedged & agg.open_coid.notna() & agg.hedge_coid.notna()].iterrows():
    oc, hc = int(r['open_coid']), int(r['hedge_coid'])
    if oc in spot_roll.index and hc in lin_roll.index:
        pairs.append((r['from_key'], oc, hc))
P = pd.DataFrame(pairs, columns=['from_key', 'o', 'h'])
P = P.join(spot_roll.add_prefix('o_'), on='o').join(lin_roll.add_prefix('h_'), on='h')

# 入场边际拆解
P['notional']  = P['o_qty'] * P['o_vwap']
P['price_pnl'] = P['o_qty'] * (P['h_vwap'] - P['o_vwap'])     # 卖perp - 买spot
P['fee_total'] = P['o_fee'] + P['h_fee']                       # +付/-返
P['edge_usdt'] = P['price_pnl'] - P['fee_total']
P['price_bps'] = P['price_pnl'] / P['notional'] * 1e4
P['fee_bps']   = -P['fee_total'] / P['notional'] * 1e4         # 负=手续费拖累
P['edge_bps']  = P['edge_usdt'] / P['notional'] * 1e4
print('matched pairs n =', len(P))
P[['from_key', 'o_vwap', 'h_vwap', 'price_bps', 'fee_bps', 'edge_bps']].head()

matched pairs n = 215


,from_key,o_vwap,h_vwap,price_bps,fee_bps,edge_bps
0,3512128730854064129,2002.05,2001.16,-4.445443,-0.999333,-5.444777
1,3512128872587984897,2002.73,2002.00,-3.645025,-0.999453,-4.644478
2,3512128881177919489,2002.73,2002.00,-3.645025,-0.999453,-4.644478
3,3512128894062821377,2002.73,2002.00,-3.645025,-0.999453,-4.644478
4,3512129091631316993,2003.44,2002.63,-4.043046,-0.999394,-5.042440


In [22]:
def stat(col):
    x = P[col]
    return {'p50': x.quantile(.5), 'mean': x.mean(), 'min': x.min(), 'max': x.max(), 'std': x.std()}

summary = pd.DataFrame({'价差 price_bps': stat('price_bps'),
                        '手续费 fee_bps': stat('fee_bps'),
                        '净边际 edge_bps': stat('edge_bps')}).T.round(3)
print('入场即时边际 (bps, 负=亏), n =', len(P))
print(summary.to_string())

print(f"\n净边际为负占比 : {(P.edge_bps < 0).mean():.1%}")
print(f"正价差占比     : {(P.price_bps > 0).mean():.1%}")
print(f"合计入场净边际 : {P.edge_usdt.sum():.4f} USDT / {P.notional.sum():.1f} USDT notional "
      f"= {P.edge_usdt.sum() / P.notional.sum() * 1e4:.2f} bps")
print(f"手续费拆解     : open(spot maker) Σ={P.o_fee.sum():.4f}  hedge(linear taker) Σ={P.h_fee.sum():.4f} USDT "
      f"(feeRate spot={P.o_feeRate.iloc[0]:+.5f} / linear={P.h_feeRate.iloc[0]:+.5f})")

# 存逐对边际表
P.to_csv(f'{SNAPSHOT}/eth_entry_edge.csv', index=False)
print(f"\nwrote {SNAPSHOT}/eth_entry_edge.csv  rows={len(P)}")

入场即时边际 (bps, 负=亏), n = 215
                p50   mean    min    max    std
价差 price_bps -4.089 -4.134 -5.038 -3.189  0.378
手续费 fee_bps  -0.999 -0.999 -1.000 -0.999  0.000
净边际 edge_bps -5.088 -5.133 -6.037 -4.188  0.378

净边际为负占比 : 100.0%
正价差占比     : 0.0%
合计入场净边际 : -4.4245 USDT / 8619.1 USDT notional = -5.13 bps
手续费拆解     : open(spot maker) Σ=-0.4310  hedge(linear taker) Σ=1.2923 USDT (feeRate spot=-0.00005 / linear=+0.00015)

wrote 20260529T030000.000000Z__20260529T040000.000000Z/eth_entry_edge.csv  rows=215


In [23]:
# 结论 (开仓即时边际):
#   - 入场净边际 mean ≈ -5.1 bps, 且 215 对 100% 为负 -> 开仓这一脚是结构性亏损。
#   - 拆解: 价差 ≈ -4.1 bps (主因) + 手续费 ≈ -1.0 bps (恒定)。
#       * 价差: 每次都买现货买在比 perp 高 ~4bps (现货比合约贵), 对冲市价卖 perp 更低,
#         锁进负基差; 0% 出现正价差 -> 不是偶发滑点, 是持续的现货升水/标的选择问题。
#       * 手续费: spot maker rebate(-0.5bp) - linear taker(+1.5bp) = 净 -1.0bp, 几乎常数。
#   - 即时边际为负不等于最终必亏: 该负基差理论上靠 持有空头 perp 的资金费 / 平仓基差收敛 来补;
#     是否真亏要看 [完整持仓盈亏] 口径 (资金费 + 平仓), 见后续分析。
#   - 若资金费/收敛补不回 ~5bp/笔, 则这是核心亏损来源: 入场基差为负 + 吃单手续费。
print('see comments above')

see comments above


## 11. SG rolling metrics 交叉验证：亏损主因是基差

SG 机器查询命令：

```bash
cd ~/rolling_metrics/bybit-margin-bybit-futures
python3 scripts/rolling_metrics/print_rolling_metrics_thresholds.py --symbol ethusdt
```

查询窗口更新时间为 `2026-05-29 08:18:58`，样本数 `14400`。关键值：

| metric | value | relevant quantiles |
|---|---:|---|
| `spread_rate` | `0.00047769` | p10 `0.00039602`, p15 `0.00041018`, p30 `0.00044116`, p70 `0.00050429`, p85 `0.00053740`, p90 `0.00055277` |
| `bidask_sr` | `0.00047272` | p10 `0.00039083`, p15 `0.00040506` |
| `askbid_sr` | `0.00048267` | p85 `0.00054242`, p90 `0.00055788` |
| `hedge_premium_rate` | `-0.00046276` | p30 `-0.00045199`, p50 `-0.00042937`, p70 `-0.00040483` |

和本 notebook 第 10 节的交易所成交结果对照：

- 实际开仓配对成交 `price_bps` 均值约 `-4.13 bps`。
- SG rolling metrics 同期 `bidask_sr`/`spread_rate` 在 `3.9-4.8 bps` 量级，`hedge_premium_rate` 为 `-4.63 bps` 量级。
- 两者量级一致，说明买现货、卖合约时锁进的负价差不是偶发执行延迟造成的，而是当时 `bybit-margin` 与 `bybit-futures` 的现货-合约基差/报价结构决定的。
- 第 7/8 节已经显示本地反应和交易所时间核对在毫秒级，不能解释稳定的 `~4 bps` 亏损；第 10 节手续费约 `-1 bps`，是稳定附加项。

结论：**ETH 这段样本的开仓即时亏损主要来自基差，约 `4 bps`；含手续费后的净 entry edge 约 `5.1 bps`。**


In [ ]:
# SG rolling_metrics cross-check numbers from print_rolling_metrics_thresholds.py --symbol ethusdt
sg = {
    'update_tp': '2026-05-29 08:18:58',
    'sample_size': 14400,
    'spread_rate': 0.00047769,
    'spread_10': 0.00039602,
    'spread_15': 0.00041018,
    'spread_30': 0.00044116,
    'spread_70': 0.00050429,
    'spread_85': 0.00053740,
    'spread_90': 0.00055277,
    'bidask_sr': 0.00047272,
    'bidask_10': 0.00039083,
    'bidask_15': 0.00040506,
    'askbid_sr': 0.00048267,
    'askbid_85': 0.00054242,
    'askbid_90': 0.00055788,
    'hedge_premium_rate': -0.00046276,
    'hedge_premium_rate_30': -0.00045199,
    'hedge_premium_rate_50': -0.00042937,
    'hedge_premium_rate_70': -0.00040483,
}
sg_bps = pd.Series(sg).drop(labels=['update_tp', 'sample_size']).astype(float) * 1e4
display(sg_bps.rename('bps').to_frame().round(3))

actual_price_bps = P['price_bps'].mean()
actual_fee_bps = P['fee_bps'].mean()
actual_edge_bps = P['edge_bps'].mean()
print(f"actual execution price_bps mean = {actual_price_bps:.3f} bps")
print(f"actual fee_bps mean             = {actual_fee_bps:.3f} bps")
print(f"actual net edge_bps mean        = {actual_edge_bps:.3f} bps")
print(f"SG bidask_sr                  = {sg['bidask_sr'] * 1e4:.3f} bps")
print(f"SG spread_rate                = {sg['spread_rate'] * 1e4:.3f} bps")
print(f"SG hedge_premium_rate         = {sg['hedge_premium_rate'] * 1e4:.3f} bps")
print('Conclusion: price loss magnitude matches SG basis metrics; fees add about -1bp on top.')


## 12. Persist aggregation

In [24]:
agg.to_parquet(OUT_PARQUET, index=False)
agg.to_csv(OUT_CSV, index=False)
print('wrote', OUT_PARQUET, '/', OUT_CSV, ' rows=', len(agg))

wrote 20260529T030000.000000Z__20260529T040000.000000Z/eth_open_hedge_agg.parquet / 20260529T030000.000000Z__20260529T040000.000000Z/eth_open_hedge_agg.csv  rows= 745


## 13. 最新 24h 订单重算：ETH 正反向执行成本

本节使用 SG `order_export_server` 重新拉取的最近 24h 总包：

`latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/uniform_orders.parquet`

相比前面 `[03:00,04:00) UTC` 单小时样本，这里 ETH 已经同时出现正向和反向：

- **forward**：`BybitMargin BUY` 开现货，`BybitFutures SELL` 对冲。price bps = `(future_sell_vwap - spot_buy_vwap) / spot_buy_vwap * 1e4`。
- **backward**：`BybitMargin SELL` 开现货，`BybitFutures BUY` 对冲。price bps = `(spot_sell_vwap - future_buy_vwap) / future_buy_vwap * 1e4`。

注意：本机当前没有 Bybit API 凭证，最新 24h 不能直接拉 `/v5/execution/list` 的真实 `execFee`。因此本节手续费沿用前面真实 execution 验证过的费率估算：spot maker `-0.5bp`，linear taker `+1.5bp`，净约 `-1bp`。如果后续在 SG 有凭证环境重新拉 execution JSON，可替换为真实 `execFee` 口径。


In [ ]:
# Latest 24h bidirectional ETH cost analysis from uniform_orders.parquet
LATEST_24H_SNAPSHOT = '20260529T033023.000000Z__20260530T033023.000000Z'
LATEST_24H_DIR = f'latest_24h/{LATEST_24H_SNAPSHOT}/{LATEST_24H_SNAPSHOT}'
LATEST_24H_SRC = f'{LATEST_24H_DIR}/uniform_orders.parquet'
LATEST_24H_PAIRS_CSV = f'{LATEST_24H_DIR}/eth_24h_open_hedge_pairs.csv'
LATEST_24H_SUMMARY_CSV = f'{LATEST_24H_DIR}/eth_24h_direction_summary.csv'

pairs24 = pd.read_csv(LATEST_24H_PAIRS_CSV)
summary24 = pd.read_csv(LATEST_24H_SUMMARY_CSV)
paired24 = pairs24[pairs24['direction'].isin(['forward', 'backward']) & (pairs24['matched_qty'] > 0)].copy()

print('latest 24h source:', LATEST_24H_SRC)
print('paired ETH groups:', len(paired24))
display(summary24.round(6))

# Distribution view by direction
dist24 = (paired24.groupby('direction')
          .agg(n=('from_key', 'count'),
               price_bps_mean=('price_bps', 'mean'),
               price_bps_p25=('price_bps', lambda x: x.quantile(.25)),
               price_bps_p50=('price_bps', 'median'),
               price_bps_p75=('price_bps', lambda x: x.quantile(.75)),
               price_bps_min=('price_bps', 'min'),
               price_bps_max=('price_bps', 'max'),
               edge_bps_est_mean=('edge_bps_est', 'mean'),
               edge_bps_est_p50=('edge_bps_est', 'median'),
               qty_mismatch=('qty_diff', lambda x: (x.abs() > 1e-9).sum()))
          .reset_index())
display(dist24.round(6))

# Direction-level headline numbers
for _, r in summary24.sort_values('direction').iterrows():
    print(
        f"{r['direction']}: n={int(r['n_pairs'])}, matched_qty={r['matched_qty']:.4f} ETH, "
        f"price_bps_wavg={r['price_bps_wavg']:.3f}, fee_bps_est={r['fee_bps_wavg_est']:.3f}, "
        f"edge_bps_est={r['edge_bps_wavg_est']:.3f}, latency_p50={r['hedge_latency_ms_p50']:.3f}ms"
    )

# Persist a compact markdown-friendly table for reports.
summary24_report = summary24[[
    'direction', 'n_pairs', 'n_qty_mismatch', 'matched_qty', 'notional',
    'price_bps_wavg', 'price_bps_p50', 'fee_bps_wavg_est', 'edge_bps_wavg_est',
    'hedge_latency_ms_p50', 'hedge_latency_ms_p99'
]].copy()
display(summary24_report.round(4))


### 24h 结论

最新 24h ETH 已同时出现正反向成交：

| direction | paired groups | matched qty | price bps | estimated fee bps | estimated edge bps |
|---|---:|---:|---:|---:|---:|
| forward (`BUY spot / SELL futures`) | 533 | 10.58 ETH | `-4.37` | `-1.00` | `-5.37` |
| backward (`SELL spot / BUY futures`) | 143 | 2.77 ETH | `+5.75` | `-1.00` | `+4.75` |

解释：

- 正向和旧单小时分析一致，买现货、卖合约时锁进负基差，price leg 约亏 `4.37bps`；加上约 `1bp` 净手续费后，entry edge 约 `-5.37bps`。
- 反向方向相反，卖现货、买合约时同一基差变成正收益，price leg 约赚 `5.75bps`；扣掉约 `1bp` 净手续费后，entry edge 约 `+4.75bps`。
- 这进一步支持第 11 节 SG rolling metrics 的判断：核心不是本地链路延迟，而是 bybit margin/futures 的基差结构。方向翻转后，基差贡献也随之翻转。
- 最新 24h 中有少量数量不完全匹配组（forward 10 组、backward 9 组），上表按 `matched_qty=min(open_qty, hedge_qty)` 计算执行成本，避免未完全对冲的残量污染 bps。


## 14. 最新 24h 全币种正反向可视化统计

本节把第 13 节 ETH 口径扩展到 `uniform_orders.parquet` 的所有币种。方向定义保持一致：

- **forward**：`BybitMargin BUY` + `BybitFutures SELL`。
- **backward**：`BybitMargin SELL` + `BybitFutures BUY`。
- 每个 `from_key` 先聚合 open/hedge 实际成交量，执行成本按 `matched_qty=min(open_qty, hedge_qty)` 计算。
- 手续费仍按估算费率：margin spot maker `-0.5bp`，linear taker `+1.5bp`，净约 `-1bp`。

最近 24h 里出现 **backward** 的币种是：`ETH`、`SOL`、`DOGE`、`DOT`。`BTC` 在这个窗口只有 forward，没有 backward。

核心结果：

| asset | direction | pairs | matched qty | notional USDT | price bps | fee bps est | edge bps est |
|---|---:|---:|---:|---:|---:|---:|---:|
| BTC | forward | 148 | 0.1480 | 10,912.89 | -4.21 | -1.00 | -5.21 |
| ETH | forward | 533 | 10.5800 | 21,263.38 | -4.37 | -1.00 | -5.37 |
| ETH | backward | 143 | 2.7700 | 5,593.79 | +5.75 | -1.00 | +4.75 |
| SOL | forward | 120 | 71.5961 | 5,915.32 | -4.07 | -1.00 | -5.07 |
| SOL | backward | 88 | 52.3779 | 4,266.87 | +6.02 | -1.00 | +5.02 |
| DOGE | forward | 79 | 39,189 | 3,912.08 | -4.42 | -1.00 | -5.42 |
| DOGE | backward | 62 | 31,322 | 3,097.02 | +5.71 | -1.00 | +4.71 |
| DOT | backward | 2 | 84.1 | 99.91 | +12.63 | -1.00 | +11.63 |

FIFO 等量抵消结果只对同时有 forward/backward 的币种计算。它回答的是：如果每个币只拿正反向共同出现的等量部分做抵消，剩余净执行成本是多少。

| asset | equal qty each side | forward edge bps | backward edge bps | combined edge bps | combined edge USDT |
|---|---:|---:|---:|---:|---:|
| ETH | 2.7700 | -5.21 | +4.75 | -0.21 | -0.2345 |
| SOL | 52.3779 | -5.10 | +5.02 | -0.07 | -0.0628 |
| DOGE | 31,322 | -5.13 | +4.71 | -0.23 | -0.1432 |

结论：其他币也出现了反向，主要是 `SOL` 和 `DOGE`，`DOT` 只有 2 笔且 notional 很小。按全量方向看，forward 普遍约 `-5bps`，backward 普遍约 `+5bps`；按 FIFO 等量抵消后，ETH/SOL/DOGE 的净亏损都收敛到约 `0.1-0.2bps`，说明之前 ETH 全量 forward `-5.37bps` 与 backward `+4.75bps` 的差异不能直接等量相加，必须先按共同成交量做 FIFO 对齐。

生成文件：

- `latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/all_symbols_24h_open_hedge_pairs.csv`
- `latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/all_symbols_24h_symbol_direction_summary.csv`
- `latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/all_symbols_24h_fifo_equal_summary.csv`


In [ ]:
# Latest 24h all-symbol bidirectional execution-cost view
ALL_24H_DIR = 'latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z'
ALL_PAIRS_CSV = f'{ALL_24H_DIR}/all_symbols_24h_open_hedge_pairs.csv'
ALL_SYMBOL_SUMMARY_CSV = f'{ALL_24H_DIR}/all_symbols_24h_symbol_direction_summary.csv'
ALL_GLOBAL_SUMMARY_CSV = f'{ALL_24H_DIR}/all_symbols_24h_global_direction_summary.csv'
ALL_FIFO_EQUAL_CSV = f'{ALL_24H_DIR}/all_symbols_24h_fifo_equal_summary.csv'

all_pairs24 = pd.read_csv(ALL_PAIRS_CSV)
all_summary24 = pd.read_csv(ALL_SYMBOL_SUMMARY_CSV)
all_global24 = pd.read_csv(ALL_GLOBAL_SUMMARY_CSV)
all_fifo24 = pd.read_csv(ALL_FIFO_EQUAL_CSV)

summary_cols = [
    'asset', 'direction', 'n_pairs', 'matched_qty', 'notional',
    'price_bps_wavg', 'fee_bps_wavg_est', 'edge_bps_wavg_est',
    'hedge_latency_ms_p50', 'hedge_latency_ms_p99'
]
print('symbols with backward:', ', '.join(all_summary24.loc[all_summary24['direction'].eq('backward'), 'asset'].tolist()))
display(all_summary24[summary_cols].sort_values(['asset', 'direction']).round(6))

fifo_cols = [
    'asset', 'equal_qty_each_side', 'forward_total_qty', 'backward_total_qty',
    'forward_edge_bps_est', 'backward_edge_bps_est',
    'combined_edge_bps_est', 'combined_edge_usdt_est'
]
display(all_fifo24[fifo_cols].round(6))


### 可视化

![matched notional by direction](latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/all_symbols_24h_direction_notional.png)

![estimated edge by direction](latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/all_symbols_24h_edge_bps_by_direction.png)

![price fee decomposition](latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/all_symbols_24h_price_fee_decomposition.png)

![FIFO equal quantity net edge](latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/all_symbols_24h_fifo_equal_edge_bps.png)

![matched pair count by direction](latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/all_symbols_24h_direction_pair_count.png)


## 15. BTC/ETH/SOL 盘口 `tp` 对齐：NEW、maker 成交、taker 成交时价差是否收敛

本节使用 SG `latency_collector` 保存的三大币盘口，不复制全量盘口文件，只在 SG 远端按订单事件时间窗口过滤必要 BBO 行。

数据来源：

- 订单事件：`latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/uniform_orders.parquet` 和 `latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/all_symbols_24h_open_hedge_pairs.csv`。
- 盘口源代码：本仓库 `src/bin/latency_collector.rs`，采集 `BTC/ETH/SOL`，venue 为 `bybit-margin` 和 `bybit-futures`。
- SG 盘口路径：`/home/ubuntu/latency_collector/data/latency_collector/<venue>/<BASE>/<YYYYMMDD_HH>.csv`。
- 盘口 CSV 字段：`local_ts_us,timestamp,venue,symbol,bid_px,bid_qty,ask_px,ask_qty`。
- 本节以盘口 CSV 的 `timestamp` 作为 `tp` 索引，对订单事件做 backward `asof` 对齐；`local_ts_us` 仅用于检查采集延迟。

订单事件阶段：

- `signal_mkt`：open order 的 `mkt_ts`，即策略看到的触发行情时间。
- `new`：open maker 订单 NEW 回报的 `update_ts`。
- `maker_fill`：open maker leg 成交回报的 `update_ts`。
- `taker_fill`：hedge taker leg 成交回报的 `update_ts`。

价差口径：

- forward：open leg 是 `BybitMargin BUY`，maker 理论成交价用 spot bid，hedge taker 用 futures bid，`maker_hedge_bps=(future_bid-spot_bid)/spot_bid*1e4`。
- backward：open leg 是 `BybitMargin SELL`，maker 理论成交价用 spot ask，hedge taker 用 futures ask，`maker_hedge_bps=(spot_ask-future_ask)/spot_ask*1e4`。
- `net` 继续扣估算 1bp fee。这个口径用来判断“maker 成交后 taker 还能不能吃到同样价差”。

生成的中间结果：

- `latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_latency_order_events.csv`：订单事件时间表。
- `latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_latency_bbo_remote_intervals.tsv`：远端 grep/awk 的时间窗口条件。
- `latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_latency_bbo_slices.csv`：远端按事件窗口过滤后的 BBO 切片。
- `latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_latency_event_bbo_snapshots.csv`：按 `tp=timestamp` asof 后的事件 BBO 快照。
- `latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_latency_stage_spread_summary.csv`：按币种/方向/阶段汇总的 BBO 价差。
- `latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_latency_threshold_fifo_summary.csv`：抬高 NEW 阈值后的成交与 FIFO 统计。


In [ ]:
# BTC/ETH/SOL latency_collector BBO diagnostics, indexed by tp=timestamp
BIG3_STAGE_CSV = f'latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_latency_stage_spread_summary.csv'
BIG3_DELTA_CSV = f'latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_latency_stage_delta_summary.csv'
BIG3_THRESH_CSV = f'latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_latency_threshold_fifo_summary.csv'
BIG3_SNAP_CSV = f'latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_latency_event_bbo_snapshots.csv'

stage15 = pd.read_csv(BIG3_STAGE_CSV)
delta15 = pd.read_csv(BIG3_DELTA_CSV)
th15 = pd.read_csv(BIG3_THRESH_CSV)
snap15 = pd.read_csv(BIG3_SNAP_CSV)

stage_cols = [
    'asset', 'direction', 'stage', 'n_with_bbo',
    'maker_hedge_bps_wavg', 'maker_hedge_net_bps_wavg_est',
    'actual_price_bps_wavg', 'actual_edge_bps_wavg_est',
    'spot_age_ms_p50', 'swap_age_ms_p50'
]
display(stage15[stage_cols].round(4))

print('NEW -> maker/taker BBO edge deltas, notional-weighted:')
delta_cols = [
    'asset', 'direction', 'n', 'notional',
    'new_maker_hedge_bps', 'maker_fill_maker_hedge_bps', 'taker_fill_maker_hedge_bps',
    'new_to_maker_delta_bps', 'maker_to_taker_delta_bps', 'new_to_taker_delta_bps',
    'actual_edge_bps_est'
]
display(delta15[delta_cols].round(4))

basis_fifo15 = th15[(th15['threshold_type'].eq('basis_magnitude_new')) &
                    (th15['direction'].eq('fifo_equal_combined'))].copy()
print('FIFO equal after raising NEW basis-magnitude threshold:')
display(basis_fifo15[['asset', 'threshold_bps', 'n_pairs', 'notional',
                      'new_basis_magnitude_bps_wavg', 'actual_edge_bps_wavg_est']].round(4))

profit15 = th15[(th15['threshold_type'].eq('profit_new_maker_hedge')) &
                (th15['direction'].ne('fifo_equal_combined'))].copy()
print('Orders whose NEW maker+taker gross edge is already positive:')
display(profit15[['asset', 'threshold_bps', 'direction', 'n_pairs', 'notional',
                  'new_maker_hedge_bps_wavg', 'actual_edge_bps_wavg_est']].round(4))

coverage = snap15['dir_maker_hedge_bps'].notna().mean()
print(f'BBO snapshot coverage: {coverage:.2%} rows have both margin and futures asof quotes')


### 盘口对齐结论

三大币的盘口诊断结论：

| asset | direction | NEW maker+taker bps | maker fill bps | taker fill bps | NEW→maker delta | actual edge bps est |
|---|---:|---:|---:|---:|---:|---:|
| BTC | forward | -3.56 | -3.73 | -3.70 | -0.18 | -5.21 |
| ETH | forward | -3.62 | -3.86 | -3.80 | -0.25 | -5.37 |
| ETH | backward | +6.17 | +5.99 | +6.14 | -0.18 | +4.75 |
| SOL | forward | -3.02 | -3.53 | -3.33 | -0.53 | -5.07 |
| SOL | backward | +7.42 | +7.16 | +7.37 | -0.46 | +5.02 |

解释：

- `NEW -> maker_fill` 期间确实有价差收敛/恶化，幅度约 `0.18-0.53bps`；SOL 最大，ETH/BTC 较小。
- `maker_fill -> taker_fill` 多数有小幅恢复，约 `0.03-0.24bps`，说明不是“maker 一成交后 taker 完全吃不到”这种单一问题。
- 主因仍是大币套利空间本身太薄：forward 从 NEW 开始就是负 edge；backward 单方向赚钱，但和 forward 做 FIFO 等量后被手续费和正反向不对称吃掉。
- 按 NEW 阶段 basis-magnitude 抬阈值后，ETH/SOL 的 FIFO 等量组合仍未转正：ETH 在 `0-3bps` 阈值约 `-0.22bps`，`4bps` 阈值约 `-0.37bps`；SOL 在 `0-3bps` 阈值约 `-0.11~-0.16bps`。没有看到“大币只要阈值拔高就能 FIFO 盈利”的证据。
- 如果按“NEW 时 maker+taker gross edge 已经为正”筛选，基本只剩 backward，forward 几乎没有同量对手盘，因此不能形成稳定双向 FIFO 盈利闭环。

所以这组数据更支持：**大币本来可捕获套利空间就很接近手续费/竞争均衡，maker 等待期间有 0.2-0.5bps 的进一步恶化，但不是唯一主因。**

![BTC maker hedge stage edge](latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_btc_maker_hedge_stage_edge.png)

![ETH maker hedge stage edge](latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_eth_maker_hedge_stage_edge.png)

![SOL maker hedge stage edge](latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_sol_maker_hedge_stage_edge.png)

![stage edge delta](latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_stage_edge_delta.png)

![basis threshold fifo edge](latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_basis_threshold_fifo_edge.png)

![profit threshold notional](latest_24h/20260529T033023.000000Z__20260530T033023.000000Z/20260529T033023.000000Z__20260530T033023.000000Z/big3_profit_threshold_notional.png)
